## speech_recognition -> Captures audio and sends to STT engines (Google, Sphinx, Whisper)
## pyaudio -> Low-level microphone and speaker I/O (required by speech_recognition)
## pyttsx3 -> Offline Text-to-Speech — no internet needed
## nltk -> Natural Langugae Processing tooklit
## openai -> CLOUD LLM 

In [1]:
!pip3 install SpeechRecognition
!pip3 install pyttsx3
!pip3 install pyaudio
!pip3 install nltk
!pip3 install openai


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pyaudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [27 lines of output]
      /private/var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/pip-build-env-hvmp083x/overlay/lib/python3.12/site-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: MIT Lice

In [4]:
!pip3  install pyaudio ## mac - run this before brew install portaudio

  Using cached PyAudio-0.2.14.tar.gz (47 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyaudio: filename=pyaudio-0.2.14-cp312-cp312-macosx_10_13_universal2.whl size=39403 sha256=a1e1c713886a7275d80199873d9fe8f883cf784de0d822c29e512cdadc6326a8
  Stored in directory: /Users/cardinality/Library/Caches/pip/wheels/68/c7/33/c6a6b210cb5819ec5c219928c794a447742a7d86d21c0b92e6
Successfully built pyaudio

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
import speech_recognition as sr
import pyaudio
import pyttsx3
import nltk
import openai 

In [6]:
 
mic_name = sr.Microphone.list_microphone_names()

if not mic_name:
    print("No microphone found. Please check your microphone settings.")
else:
    print("Available microphones:")
    for index, name in enumerate(mic_name):
        print(f"{index}: {name}")

Available microphones:
0: AB13X USB Audio
1: AB13X USB Audio
2: MacBook Air Microphone
3: MacBook Air Speakers
4: Microsoft Teams Audio


In [8]:
def listen():
    r = sr.Recognizer()
    with sr.Microphone() as source:
        print("Listening...")
        audio = r.listen(source)
    try:
        text = r.recognize_google(audio)
        print(f"You said: {text}")
        return text
    except sr.UnknownValueError:
        print("Sorry, I could not understand the audio.")
        return None
    except sr.RequestError as e:
        print(f"Could not request results; {e}")
        return None
    
def process(text):
    # Placeholder for NLP processing
    return "I am good thank you! How can I assist you today?"

def speak(text):
    engine = pyttsx3.init()
    engine.say(text)
    engine.runAndWait()


def run_bot():
    user_input = listen()
    print(f"User input: {user_input}")
    if user_input:
        response = process(user_input)
        print(f"Bot response: {response}")
        speak(response)
    
if __name__ == "__main__":
    run_bot()


Listening...
You said: hello hi how are you
User input: hello hi how are you
Bot response: I am good thank you! How can I assist you today?


In [ ]:
## Speech to Text - Deep Dive

In [11]:
import speech_recognition as sr

# The Recognizer is the core object — it holds config and methods
recognizer = sr.Recognizer()

print("Recognizer Thresholds:")
print(f"Energy threshold: {recognizer.energy_threshold}") ## Sensitivity - Low Value hears BG noise , High  Value misses low volume speech
print(f"Dynamic energy threshold: {recognizer.dynamic_energy_threshold}") ## Auto adjust energy threshold based on ambient noise
print(f"Pause threshold: {recognizer.pause_threshold}") ## Seconds of silence to consider end of a phrase
print(f"Adhjust for ambient noise: {recognizer.adjust_for_ambient_noise}") ## Adjust energy threshold based on ambient noise for a duration

Recognizer Thresholds:
Energy threshold: 300
Dynamic energy threshold: True
Pause threshold: 0.8
Adhjust for ambient noise: <bound method Recognizer.adjust_for_ambient_noise of <speech_recognition.Recognizer object at 0x14eb716d0>>


In [ ]:
## WaitTimeoutError — user didn't speak in time, ask them again
## UnknownValueError — something was detected but couldn't be decoded, ask again
## RequestError — network problem, handle gracefully

In [12]:
import pyttsx3
import speech_recognition as sr
import tempfile,os,subprocess


## Generate Audio from Text

def text_to_wav(text, filename):
    engine = pyttsx3.init()
    engine.setProperty('rate', 150)  # Speed of speech
    engine.save_to_file(text, filename)
    engine.runAndWait()
    engine.stop()

def ensure_pcm_wav(input_path:str, output_path:str):
    """Convert input audio to PCM WAV format using ffmpeg."""
    try:
        from pydub import AudioSegment
        audio = AudioSegment.from_file(input_path)
        audio = audio.set_frame_rate(16000).set_channels(1).set_sample_width(2) # 16kHz, mono, 16-bit
        audio.export(output_path, format="wav")
        return True
    except ImportError:
        print("pydub not available, falling back to ffmpeg for conversion.")
    except Exception as e:
        print(f"pydub conversion failed: {e}. Falling back to ffmpeg.")

    try:

        command = [
            'ffmpeg',
            '-i', input_path,
            '-acodec', 'pcm_s16le',
            '-ar', '16000',
            '-ac', '1',
            output_path
        ]
        result =subprocess.run(command,  capture_output=True, text=True,check=True)
        if result.returncode != 0:
            print(f"ffmpeg error: {result.stderr}")
            return False
        return True 
    except Exception as e:
        print(f"ffmpeg conversion failed: {e}")
        return False
    

def wav_to_text(wav_path,language="en-IN"):
    recognizer = sr.Recognizer()
    with sr.AudioFile(wav_path) as source:
        audio = recognizer.record(source)
    try:
        text = recognizer.recognize_google(audio,language=language)
        return text
    except sr.UnknownValueError:
        print("Google Speech Recognition could not understand audio")
        return None
    except sr.RequestError as e:
        print(f"Could not request results from Google Speech Recognition service; {e}")
        return None


original_text = "Hello, how are you doing today? This is a test of the text to speech and speech to text conversion." \

tmp_dir = tempfile.gettempdir()
wav_path = os.path.join(tmp_dir, "output.wav")
pcm_wav_path = os.path.join(tmp_dir, "output_pcm.wav")

print(f"Original Text: {original_text}")
text_to_wav(original_text, wav_path)
print(f"Generated WAV file at: {wav_path}")
size = os.path.getsize(wav_path) if os.path.exists(wav_path) else 0
print(f"  Saved raw audio: {wav_path} ({size/1024:.1f} KB)")


success = ensure_pcm_wav(wav_path, pcm_wav_path)

if success and os.path.exists(pcm_wav_path):
    print(f"Converted to PCM WAV: {pcm_wav_path}")
    size = os.path.getsize(pcm_wav_path)
    print(f"  PCM WAV file: {pcm_wav_path} ({size/1024:.1f} KB)")
    recognized_text = wav_to_text(pcm_wav_path)
    print(f"Recognized Text: {recognized_text}")
else:
    print("Failed to convert to PCM WAV. Cannot proceed with speech recognition.")      






Original Text: Hello, how are you doing today? This is a test of the text to speech and speech to text conversion.
Generated WAV file at: /var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/output.wav
  Saved raw audio: /var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/output.wav (298.0 KB)
Converted to PCM WAV: /var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/output_pcm.wav
  PCM WAV file: /var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/output_pcm.wav (213.4 KB)
Recognized Text: hello how are you doing today this is a test of the text to speech and speech to text conversion


## Text to Speech ->



In [13]:
import pyttsx3


engine = pyttsx3.init()


# List all installed voices
voices = engine.getProperty('voices')

for i, voice in enumerate(voices):
    short_id = voice.id.split("\\")[-1].split("/")[-1][:28]
    print(f"{i:<6} {short_id:<30} {voice.name}")

0      com.apple.speech.synthesis.v   Albert
1      com.apple.voice.compact.it-I   Alice
2      com.apple.voice.compact.sv-S   Alva
3      com.apple.voice.compact.fr-C   Amélie
4      com.apple.voice.compact.ms-M   Amira
5      com.apple.voice.compact.de-D   Anna
6      com.apple.speech.synthesis.v   Bad News
7      com.apple.speech.synthesis.v   Bahh
8      com.apple.speech.synthesis.v   Bells
9      com.apple.speech.synthesis.v   Boing
10     com.apple.speech.synthesis.v   Bubbles
11     com.apple.voice.compact.he-I   Carmit
12     com.apple.speech.synthesis.v   Cellos
13     com.apple.voice.compact.id-I   Damayanti
14     com.apple.voice.compact.en-G   Daniel
15     com.apple.voice.compact.bg-B   Daria
16     com.apple.speech.synthesis.v   Wobble
17     com.apple.eloquence.de-DE.Ed   Eddy (German (Germany))
18     com.apple.eloquence.en-GB.Ed   Eddy (English (UK))
19     com.apple.eloquence.en-US.Ed   Eddy (English (US))
20     com.apple.eloquence.es-ES.Ed   Eddy (Spanish (Spain))
2

In [ ]:
## Rate - words per minute, default 200 which is fast — 150 is more natural for a bot
## Volume - 0.0 to 1.0, default 1.0
## Voice - depends on OS, use getProperty('voices') to list available voices and their IDs, then set with setProperty('voice', voice_id)

In [1]:
import pyttsx3

engine = pyttsx3.init()
sentence = "Hello, how are you doing today? This is a test of the text to speech conversion."
rate = 200
engine.setProperty('rate', rate)
print(f"Speaking at {rate} WPM")
engine.say(sentence)
engine.runAndWait()
rate = 150
engine.setProperty('rate', rate)
print(f"Speaking at {rate} WPM")
engine.say(sentence)
engine.runAndWait()


engine.stop()

Speaking at 200 WPM
Speaking at 150 WPM


In [ ]:
## https://developers.openai.com/api/docs/guides/text-to-speech

# Show all available voices
openai_voices = {
    "alloy"  : "Neutral, balanced",
    "echo"   : "Smooth, male",
    "fable"  : "Expressive, British",
    "onyx"   : "Deep, authoritative",
    "nova"   : "Friendly, female",
    "shimmer": "Gentle, clear",
}

from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
import os

from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

In [4]:
response = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="alloy",
    input="Hello, how are you doing today? This is a test of the text to speech conversion using OpenAI's TTS model."
)

response.stream_to_file("openai_tts_output.wav")
print("OpenAI TTS output saved to openai_tts_output.wav")

OpenAI TTS output saved to openai_tts_output.wav


/var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/ipykernel_12918/3385356335.py:7: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file("openai_tts_output.wav")


In [1]:
import pyttsx3

class TTSEngine:

    def __init__(self,rate = 150, volume = 1.0, voice_id = None):
        self.engine = pyttsx3.init()
        self.set_rate(rate)
        self.set_volume(volume)
        self.set_voice(voice_id)


    def set_rate(self, rate):
        self.engine.setProperty('rate', rate)

    def set_volume(self, volume):
        self.engine.setProperty('volume', volume)

    def set_voice(self, voice_id):
        self.engine.setProperty('voice', voice_id)

        

    def speak(self, text,filename):
        self.engine.say(text)
        self.engine.runAndWait()
        self.engine.save_to_file(text, filename)



tts_engine = TTSEngine(rate=150, volume=0.9, voice_id=None)
# tts_engine.speak("Hello, how are you doing today? This is a test of the text to speech conversion using the Engine class.", "engine_tts_output.wav")


In [2]:
tts_engine.set_rate(200)
tts_engine.speak("Now speaking at a faster rate of 200 words per minute.", "engine_tts_output_fast.wav")

In [ ]:
## Intent and bussiness logic would go here, using the TTS engine to generate responses and the STT engine to capture user input.


## Intent are are based on two approaches:
## 1. Rule-based: Define specific patterns or keywords to identify intents (e.g., "book a flight", "what's the weather")
## 2. LLM BASED : USE GPT/CLAUDE TO CLASSIFY INTENT BASED ON TRAINING EXAMPLES OR ZERO-SHOT PROMPTING. THIS CAN CAPTURE MORE NUANCED INTENTS BUT MAY REQUIRE MORE COMPUTE AND CAREFUL PROMPT DESIGN.

In [1]:

INTENTS = {
    'greeting': [
        'hello', 'hi', 'hey', 'good morning', 'good afternoon',
        'good evening', 'howdy', 'what\'s up', 'namaste'
    ],
    'farewell': [
        'bye', 'goodbye', 'see you', 'exit', 'quit',
        'stop', 'close', 'that\'s all', 'thank you bye'
    ],
    'get_time': [
        'time', 'what time', 'clock', 'current time', 'tell me the time'
    ],
    'get_date': [
        'date', 'what date', 'today\'s date', 'what day', 'which day'
    ],
    'calculate': [
        'calculate', 'compute', 'math', 'plus', 'minus', 'times',
        'divided', 'multiply', 'add', 'subtract', 'sum', 'total'
    ],
    'tell_joke': [
        'joke', 'funny', 'make me laugh', 'tell me something funny', 'humor'
    ],
    'help': [
        'help', 'what can you do', 'commands', 'options', 'capabilities'
    ],
}

In [2]:
def classify_intent(user_input:str, intents :dict = INTENTS):
    user_input = user_input.lower()
    for intent, keywords in intents.items():
        for keyword in keywords:
            if keyword in user_input:
                return intent
    return "unknown"

In [3]:
test_phrases = [
    "Hello there!",
    "What time is it right now?",
    "Calculate 15 plus 27",
    "Tell me a joke please",
    "What's today's date?",
    "Goodbye, see you later!",
    "Play some music",       # unknown
    "What's the weather?",   # unknown
]


for phrase in test_phrases:
    intent = classify_intent(phrase)
    print(f"Input: '{phrase}' => Classified Intent: '{intent}'")

Input: 'Hello there!' => Classified Intent: 'greeting'
Input: 'What time is it right now?' => Classified Intent: 'get_time'
Input: 'Calculate 15 plus 27' => Classified Intent: 'calculate'
Input: 'Tell me a joke please' => Classified Intent: 'tell_joke'
Input: 'What's today's date?' => Classified Intent: 'get_date'
Input: 'Goodbye, see you later!' => Classified Intent: 'farewell'
Input: 'Play some music' => Classified Intent: 'unknown'
Input: 'What's the weather?' => Classified Intent: 'unknown'


In [4]:
from datetime import datetime
import random,re

def handle_greeting():
    responses = ["Hello! How can I assist you today?", "Hi there! What can I do for you?", "Hey! Need any help?"]
    return random.choice(responses)


def handle_farewell():
    responses = ["Goodbye! Have a great day!", "See you later! Take care!", "Bye! Feel free to come back anytime!"]
    return random.choice(responses)

def handle_get_time():
    now = datetime.now()
    return f"The current time is {now.strftime('%I:%M %p')}."

def handle_get_date():
    today = datetime.today()
    return f"Today's date is {today.strftime('%B %d, %Y')}."

def handle_calculate(user_input):
    # Extract the math expression using regex
    expression = re.findall(r'calculate (.+)', user_input)
    if not expression:
        return "Sorry, I couldn't find a math expression to calculate."
    
    try:
        # WARNING: eval can be dangerous, use with caution and proper sanitization in production
        result = eval(expression[0])
        return f"The result of {expression[0]} is {result}."
    except Exception as e:
        return f"Sorry, I couldn't calculate that. Error: {e}"
    
def handle_tell_joke():
    jokes = [
        "Why don't scientists trust atoms? Because they make up everything!",
        "Why did the math book look sad? Because it had too many problems.",
        "What do you call fake spaghetti? An impasta!"
    ]
    return random.choice(jokes)

def handle_help():
    return "I can help you with greetings, farewells, telling the time and date, simple calculations, and jokes. Just ask!"

def handle_unknown():
    return "Sorry, I didn't understand that. Can you please rephrase?"

In [7]:
def process(user_input):
    intent = classify_intent(user_input)
    if intent == 'greeting':
        response = handle_greeting()
        return intent,response
    elif intent == 'farewell':
        response = handle_farewell()
        return intent,response
    elif intent == 'get_time':
        response = handle_get_time()
        return intent,response
    elif intent == 'get_date':
        response = handle_get_date()
        return intent,response
    elif intent == 'calculate':
        response = handle_calculate()
        return intent,response
    elif intent == 'tell_joke':
        response = handle_tell_joke()
        return intent,response
    elif intent == 'help':
        response = handle_help()
        return intent,response
    else:
        response = handle_unknown()
        return intent,response

In [6]:
## STT -> Intent Classification -> Response Generation -> TTS
import speech_recognition as sr
import pyttsx3
recognizer = sr.Recognizer()
try:
    with sr.Microphone() as source:
        print("Listening...")
        audio = recognizer.listen(source)
    user_input = recognizer.recognize_google(audio)
    print(f"You said: {user_input}")
except sr.UnknownValueError:
    print("Sorry, I could not understand the audio.")
    user_input = None
except sr.RequestError as e:
    print(f"Could not request results; {e}")
    user_input = None
if user_input:
    intent,handler_response = process(user_input)
    print(f"Intent: {intent}, Response: {handler_response}")

    engine = pyttsx3.init()
    engine.setProperty('rate', 150)
    engine.setProperty('volume', 0.9)
    engine.say(handler_response)
    engine.runAndWait()
    engine.stop()

Listening...
You said: hello hi how are you
Intent: greeting, Response: Hi there! What can I do for you?


In [ ]:
## 1. Modify Intent based on LLM  usign Open AI models
## 2. TTS to use OpenAI's TTS models for more natural speech synthesis. instead of pyttsx3